# TP4 - Redes convolucionales para clasificar lesiones de piel (ISIC)

**Dataset:** imágenes dermatoscópicas ISIC de 9 clases (~880 imágenes en la carpeta `data/Split_smol`, con una subcarpeta por clase dentro de `train/` y `val/`).

El **código ya está resuelto y probado**. Lo que se evalúa es **su análisis**. El cómputo lleva unos 15-20 minutos (medido en una CPU de 20 núcleos), y les queda la mayor parte de las 2 horas para interpretar resultados.

**Qué se trabaja en este TP:** análisis exploratorio, limpieza de duplicados y split, convoluciones, pooling y campo receptivo, comparación MLP vs. CNN, decisiones de arquitectura, resolución de entrada, data augmentation y evaluación final.
**Qué NO se repite** porque ya lo hicieron en los TP1 a TP3: búsqueda de hiperparámetros con Optuna, dropout/L2/BatchNorm como variables de estudio (la BatchNorm queda fija dentro del bloque convolucional), inicialización y logging con TensorBoard/MLflow.

**Cómo trabajar:** ejecuten de arriba hacia abajo, lean los outputs y completen las celdas **✏️** (observaciones de EDA, explicación del código convolucional y cuatro análisis). Las comparaciones usan **3 semillas** y reportan media ± desvío: una diferencia solo cuenta si supera ese ruido.

**Reglas metodológicas (las mismas de los TPs anteriores):**
- Las entradas van **normalizadas** (media y desvío por canal calculados **solo en train**).
- El **test se usa una sola vez**, en la sección 8. Todo lo demás se decide con validación.
- La augmentation se aplica **solo a train y después del split**.
- Las conclusiones tienen que **coincidir con los outputs** de su corrida.

## 0. Configuración
Si no tienen GPU, los valores por defecto funcionan en CPU. Si cambian alguna constante, **anótenlo en su análisis**.

In [1]:
import os, time, random, copy, hashlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import warnings; warnings.filterwarnings("ignore")

DATA_DIR    = "data/Split_smol"    # carpeta con train/ y val/ (una subcarpeta por clase)
SIZES       = [32, 64, 96]         # resoluciones de entrada que se van a comparar
SEEDS       = [0, 1, 2]            # semillas para medir ruido
MAX_EPOCHS  = 60                   # épocas máximas (con early stopping)
PATIENCE    = 10                   # épocas sin mejorar val_loss antes de cortar
BATCH       = 32
AUG_COPIES  = 2                    # copias aumentadas que se agregan a cada imagen de train
SEED_SPLIT  = 42                   # semilla del split (NO cambiarla entre corridas)
SEEDS_FINAL = [0, 1, 2, 3, 4]      # semillas de la evaluación final

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print("Dispositivo:", device, "| torch", torch.__version__)

Dispositivo: cpu | torch 2.14.1+cpu


## 1. Análisis exploratorio (EDA)
Antes de modelar hay que mirar los datos: cuántas imágenes hay por clase, de qué tamaños son, cómo se ven y si hay algo raro (desbalance, resoluciones distintas por clase, **imágenes repetidas**). Acá todavía no se divide nada: se trabaja con todas las imágenes tal como vienen.

In [2]:
class_names = sorted(os.listdir(f"{DATA_DIR}/train")); NUM_CLASSES = len(class_names)
meta = []
for sp in ["train", "val"]:
    for ci, c in enumerate(class_names):
        for f in sorted(os.listdir(f"{DATA_DIR}/{sp}/{c}")):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                p = f"{DATA_DIR}/{sp}/{c}/{f}"
                with Image.open(p) as im: w, h = im.size
                meta.append(dict(path=p, carpeta=sp, clase=c, y=ci, w=w, h=h, md5=hashlib.md5(open(p, "rb").read()).hexdigest()))
df = pd.DataFrame(meta); df["res"] = df["w"].astype(str) + "x" + df["h"].astype(str); df["aspecto"] = df["w"] / df["h"]
print(f"{len(df)} imágenes, {NUM_CLASSES} clases")

# Se cargan todas las imágenes una sola vez, ya redimensionadas a cuadrados, para todas las resoluciones
t0 = time.time(); RAW = {s: [] for s in SIZES}
for p in df["path"]:
    with Image.open(p) as im0:
        im = im0.convert("RGB")
        for s in SIZES: RAW[s].append(np.asarray(im.resize((s, s), Image.BILINEAR)))
X_U8 = {s: torch.from_numpy(np.stack(RAW[s])).permute(0, 3, 1, 2).contiguous() for s in SIZES}; del RAW
Y = torch.from_numpy(df["y"].values); labels = df["y"].values
print(f"Carga de imágenes: {time.time() - t0:.0f}s | tensor {tuple(X_U8[64].shape)} uint8")

FileNotFoundError: [Errno 2] No such file or directory: 'data/Split_smol/train'

In [ ]:
# 1.1 Clases y carpeta de origen
display(pd.crosstab(df["clase"], df["carpeta"], margins=True))
# 1.2 Resoluciones: ¿son todas iguales? ¿dependen de la clase?
print("Resoluciones más frecuentes:"); display(df["res"].value_counts().head(8).to_frame("imágenes").T)
top = df["res"].value_counts().head(5).index
ct = pd.crosstab(df["clase"], df["res"].where(df["res"].isin(top), "otras"), normalize="index").round(2)
print("Fracción de cada clase por resolución:"); display(ct)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
df["aspecto"].hist(bins=30, ax=ax[0]); ax[0].set_title("Relación de aspecto (ancho/alto) - se redimensiona a cuadrado")
(df.groupby("clase")["w"].median()).plot.barh(ax=ax[1]); ax[1].set_title("Ancho mediano en píxeles por clase"); plt.tight_layout(); plt.show()

### Resoluciones
El dataset tiene 878 imágenes en 9 clases y está casi balanceado: siete clases tienen 100 imágenes, Atopic Dermatitis tiene 102 y Tinea Ringworm Candidiasis es la más chica con 76. En la partición original, train tenía 697 imágenes y val 181, y val estaba perfectamente balanceada (20 por clase, 21 en Atopic Dermatitis), mientras que train arrastraba el desbalance de Tinea (56 contra 80). Después de eliminar duplicados el desbalance se acentuó un poco: Atopic Dermatitis quedó en 80 imágenes y Tinea en 64, contra 100 de las demás. Quedaron 844 imágenes y el split estratificado dio train 506 / val 169 / test 169. En val y test, Tinea tiene solo 13 imágenes y Atopic 16, así que las métricas de esas clases son muy poco estables.

In [ ]:
# 1.3 Color medio por clase y ejemplos
mean_rgb = pd.DataFrame(X_U8[32].float().mean((2, 3)).numpy() / 255, columns=["R", "G", "B"]); mean_rgb["clase"] = df["clase"].values
mean_rgb.groupby("clase").mean().plot.barh(color=["#c44e52", "#55a868", "#4c72b0"], figsize=(8, 4), title="Color medio (RGB) por clase"); plt.tight_layout(); plt.show()
fig, axes = plt.subplots(NUM_CLASSES, 6, figsize=(11, 1.9 * NUM_CLASSES))
for ci in range(NUM_CLASSES):
    ids = np.where(labels == ci)[0][:6]
    for j, i in enumerate(ids): axes[ci, j].imshow(X_U8[96][i].permute(1, 2, 0)); axes[ci, j].axis("off")
    axes[ci, 0].set_title(class_names[ci], fontsize=8, loc="left")
plt.tight_layout(); plt.show()

### 1.4 Imágenes duplicadas
El dataset viene con una partición `train/`/`val/` propia. Si una misma imagen está en las dos carpetas, la validación **no mide generalización** (el modelo ya vio esas imágenes). Se buscan duplicados **exactos** (mismo hash MD5 del archivo) y se verifica con un hash perceptual (diferencias entre píxeles vecinos de una miniatura) que no queden **casi-duplicados** (recortes o recompresiones).

In [ ]:
g = df.groupby("md5")
dup_groups = [idx for _, idx in g.groups.items() if len(idx) > 1]
cross = [idx for idx in dup_groups if df.loc[idx, "carpeta"].nunique() > 1]
print(f"Grupos de imágenes idénticas: {len(dup_groups)} (de los cuales {len(cross)} están repetidos entre train y val)")
print("Por clase:", df.loc[[idx[0] for idx in dup_groups], "clase"].value_counts().to_dict())
print("Conflictos de etiqueta (misma imagen con clases distintas):", sum(df.loc[idx, "y"].nunique() > 1 for idx in dup_groups))

# hash perceptual: miniatura gris 16x17, bit = pixel > vecino derecho (256 bits); distancia de Hamming
gray = X_U8[32].float().mean(1, keepdim=True)
th_ = F.interpolate(gray, size=(16, 17), mode="area")
H = (th_[..., 1:] > th_[..., :-1]).flatten(1).numpy()
near = []
for i in range(len(H)):
    d = (H[i] != H[i + 1:]).sum(1)
    near += [(i, i + 1 + j) for j in np.where(d <= 8)[0]]
near_not_exact = [(i, j) for i, j in near if df.loc[i, "md5"] != df.loc[j, "md5"]]
print(f"Pares con hash perceptual casi igual (<=8 bits de 256): {len(near)} | de ellos NO idénticos por MD5: {len(near_not_exact)}")

### ✏️ TUS OBSERVACIONES DEL EDA
Respondan en pocas líneas, con los números de los outputs de arriba:
1. ¿Qué tan desbalanceado está el dataset? ¿Qué cambia respecto de la partición original `train`/`val`?
2. ¿Hay clases que se distinguen por resolución, relación de aspecto o color de fondo? ¿Qué riesgo tiene eso para un modelo que ve las imágenes redimensionadas?
3. ¿Qué duplicados encontraron, en qué clases y por qué importan para la validación? ¿Qué hubiera pasado con la `val_acc` si no los eliminaran?

-1. El dataset tiene 878 imágenes en 9 clases y está casi balanceado: siete clases tienen 100 imágenes, Atopic Dermatitis tiene 102 y Tinea Ringworm Candidiasis es la más chica con 76. En la partición original, train tenía 697 imágenes y val 181, y val estaba perfectamente balanceada (20 por clase, 21 en Atopic Dermatitis), mientras que train arrastraba el desbalance de Tinea (56 contra 80). Después de eliminar duplicados el desbalance: Atopic Dermatitis quedó en 80 imágenes y Tinea en 64, contra 100 de las demás. Quedaron 844 imágenes y el split estratificado dio train 506 / val 169 / test 169. En val y test, Tinea tiene solo 13 imágenes y Atopic 16, así que las métricas de esas clases son muy poco estables.

-2. La resolución casi identifica el origen de la imagen y, con él, la clase. Todas las imágenes de Atopic Dermatitis son de 294x222, y todas las de Actinic keratosis, Dermatofibroma, Squamous cell carcinoma y Vascular lesion son de 600x450. Melanocytic nevus (79% en 1024x768) y Melanoma (56%) son casi todas de ~1024 px, y Benign keratosis tiene 74% en 1024x764. Tinea no tiene ninguna de las cinco resoluciones más comunes. La relación de aspecto es casi siempre ~1.33 (ancho/alto), así cuando se pasa todo a cuadrado se deforma de manera parecida en todas las clases.
En color también hay diferencias. Atopic Dermatitis tiene el azul más bajo (~0.38) y Tinea también (~0.47). Melanoma es más oscura (R ~0.71, B ~0.53), mientras que Vascular, Squamous y Dermatofibroma son más claras y rosadas (R ~0.79, B ~0.62-0.64). En la galería se ve que Atopic Dermatitis son fotos de brazos y piernas, no dermatoscopías, y que Melanoma y nevus suelen tener viñeteado negro o fondo celeste. Una parte de estas diferencias de color es clínica (la inflamación o los vasos se ven más rojos), pero otra parte depende del equipo y la iluminación. El riesgo es que la red aprenda la fuente de la imagen (fondo, viñeteado, nitidez, color) en vez de la lesión. Aunque redimensionemos, esas pistas quedan, y como val y test salen de las mismas fuentes, el accuracy puede quedar inflado y no generalizaría a imágenes de otro hospital o de otro equipo.

-3. Se encontraron 34 grupos de imágenes idénticas (mismo MD5), y 33 de ellos estaban repetidos entre train y val. Aparecen solo en dos clases: Atopic Dermatitis (22) y Tinea (12). No hay conflictos de etiqueta y el hash perceptual no encontró casi duplicados. Importan porque la validación deja de medir generalización. Es decir, el modelo ya vio esas imágenes en train. En Atopic Dermatitis casi toda la val original (21 imágenes) estaba en train, y en Tinea 12 de 20. En total, alrededor de un 18% de la val original (33 de 181) eran copias de train. No medimos cuánto habría subido la val_acc, pero con una cuenta gruesa la inflaría en unos 6 o 7 puntos, y además las dos clases más débiles se verían mejor de lo que son. Por eso se elimina una copia de cada grupo antes de rehacer el split.

## 2. Eliminación de duplicados y split estratificado
- Se **elimina una copia de cada imagen duplicada** (se conserva una sola por grupo de MD5).
- Se juntan **todas** las imágenes que quedan (la partición original no sirve: tenía repetidas y no tenía test) y se hace un split **estratificado 60/20/20** con semilla fija. El test no se toca hasta la sección 8.
- Se calcula la **normalización** (media y desvío por canal) **solo con train**.

In [ ]:
keep = df.groupby("md5").head(1).index.values            # una imagen por hash
print(f"Imágenes totales: {len(df)} -> sin duplicados: {len(keep)} (se eliminaron {len(df) - len(keep)})")
idx_tr, idx_tmp = train_test_split(keep, test_size=0.4, stratify=labels[keep], random_state=SEED_SPLIT)
idx_va, idx_te  = train_test_split(idx_tmp, test_size=0.5, stratify=labels[idx_tmp], random_state=SEED_SPLIT)
print(f"train {len(idx_tr)} | val {len(idx_va)} | test {len(idx_te)}")
print(pd.DataFrame({"train": pd.Series(df.loc[idx_tr, "clase"]).value_counts(), "val": pd.Series(df.loc[idx_va, "clase"]).value_counts(),
                    "test": pd.Series(df.loc[idx_te, "clase"]).value_counts()}).to_string())
assert not (set(df.loc[idx_tr, "md5"]) & set(df.loc[idx_va, "md5"])) and not (set(df.loc[idx_tr, "md5"]) & set(df.loc[idx_te, "md5"])) and not (set(df.loc[idx_va, "md5"]) & set(df.loc[idx_te, "md5"]))
STATS = {}
for s in SIZES:
    xt = X_U8[s][idx_tr].float() / 255
    STATS[s] = (xt.mean((0, 2, 3)).view(1, 3, 1, 1), xt.std((0, 2, 3)).view(1, 3, 1, 1))
print("mean/std por canal (64px, solo train):", STATS[64][0].flatten().numpy().round(3), STATS[64][1].flatten().numpy().round(3))

## 3. Data augmentation a mano (solo sobre train, después del split)
No se usa ninguna librería de augmentation: las transformaciones están escritas con operaciones de tensores. Cada imagen de **train** se complementa con `AUG_COPIES` copias transformadas **ya generadas** (se calculan una vez, después del split, así que ninguna copia de una imagen de val/test puede filtrarse a train). Val y test **nunca** se aumentan.

Transformaciones disponibles: flips horizontal/vertical, rotaciones de 90° y brillo/contraste aleatorios. Las lesiones dermatoscópicas no tienen orientación privilegiada, pero el **color** es información diagnóstica.

In [ ]:
AUGS = {"ninguna": {}, "flips": dict(hflip=True, vflip=True),
        "flips+rot90": dict(hflip=True, vflip=True, rot90=True),
        "flips+rot90+color": dict(hflip=True, vflip=True, rot90=True, color=0.2)}

def augment(x, a):
    # x: tensor float [B,3,H,W] en [0,1]. Cada imagen recibe sus propias transformaciones al azar.
    B = x.shape[0]; x = x.clone()
    if a.get("hflip"):
        m = torch.rand(B) < 0.5; x[m] = x[m].flip(3)
    if a.get("vflip"):
        m = torch.rand(B) < 0.5; x[m] = x[m].flip(2)
    if a.get("rot90"):
        k = torch.randint(0, 4, (B,))
        for kv in (1, 2, 3):
            m = k == kv
            if m.any(): x[m] = torch.rot90(x[m], kv, (2, 3))
    if a.get("color"):
        c = a["color"]
        bri = 1 + (torch.rand(B, 1, 1, 1) * 2 - 1) * c; con = 1 + (torch.rand(B, 1, 1, 1) * 2 - 1) * c
        mu = x.mean((1, 2, 3), keepdim=True); x = ((x - mu) * con + mu) * bri
    return x.clamp(0, 1)

TRAIN_SETS = {}
def train_set(size, aug_name):
    # originales de train + AUG_COPIES copias aumentadas de cada una (generadas una vez, con semilla fija)
    key = (size, aug_name)
    if key not in TRAIN_SETS:
        x0, y0 = X_U8[size][idx_tr], Y[idx_tr]
        if aug_name == "ninguna": TRAIN_SETS[key] = (x0, y0)
        else:
            set_seed(1234); xs, ys = [x0], [y0]
            for _ in range(AUG_COPIES): xs.append((augment(x0.float() / 255, AUGS[aug_name]) * 255).round().to(torch.uint8)); ys.append(y0)
            TRAIN_SETS[key] = (torch.cat(xs), torch.cat(ys))
    return TRAIN_SETS[key]

# Vista de una imagen y sus versiones aumentadas
i0 = idx_tr[0]; base = X_U8[96][i0:i0 + 1].float() / 255; set_seed(3)
fig, ax = plt.subplots(1, 6, figsize=(12, 2.3)); ax[0].imshow(base[0].permute(1, 2, 0)); ax[0].set_title("original")
for k in range(1, 6): ax[k].imshow(augment(base, AUGS["flips+rot90+color"])[0].permute(1, 2, 0)); ax[k].set_title("aumentada")
for a in ax: a.axis("off")
plt.show()

## 4. Modelos y entrenamiento
- **`MLP`**: la referencia (Flatten + 2 capas ocultas).
- **`CNN`**: `n_blocks` bloques `Conv2d → BatchNorm → ReLU → MaxPool(2)` con `filters·2^b` canales, y una cabeza `Flatten → Linear(128) → ReLU → Dropout → Linear(clases)`.
- **`rf_full(...)`** calcula el **campo receptivo** teórico de la última capa convolucional.
- **Entrenamiento**: Adam, early stopping sobre `val_loss` restaurando el mejor estado, tiempo por época medido.

In [ ]:
class MLP(nn.Module):
    def __init__(self, size, num_classes=NUM_CLASSES):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(3 * size * size, 512), nn.ReLU(),
                                 nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, num_classes))
    def forward(self, x): return self.net(x)

class CNN(nn.Module):
    def __init__(self, size, n_blocks=3, filters=16, kernel=3, use_bn=True, dropout=0.2, num_classes=NUM_CLASSES):
        super().__init__()
        layers, cin = [], 3
        for b in range(n_blocks):
            cout = filters * 2 ** b
            layers += [nn.Conv2d(cin, cout, kernel, padding=kernel // 2, bias=not use_bn)]
            if use_bn: layers += [nn.BatchNorm2d(cout)]
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            cin = cout
        self.features = nn.Sequential(*layers)
        fs = size // 2 ** n_blocks                       # lado del mapa de activación después de los n_blocks pooling
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(cin * fs * fs, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, x): return self.head(self.features(x))

def rf_full(n_blocks, kernel):
    # campo receptivo de la última conv: cada conv suma (k-1)*salto; cada maxpool(2) suma 1*salto y duplica el salto
    r, j = 1, 1
    for b in range(n_blocks):
        r += (kernel - 1) * j
        if b < n_blocks - 1: r += 1 * j; j *= 2
    return r
def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

def norm_batch(x_u8, size):
    mu, sd = STATS[size]; return ((x_u8.float() / 255 - mu) / sd).to(device)

def get_eval_batch(size, ids, shift=0):
    x = X_U8[size][ids].float() / 255
    if shift:   # traslación circular aleatoria (solo para medir robustez a traslaciones)
        x = torch.stack([torch.roll(xi, shifts=(int(np.random.randint(-shift, shift + 1)), int(np.random.randint(-shift, shift + 1))), dims=(1, 2)) for xi in x])
    mu, sd = STATS[size]; return ((x - mu) / sd).to(device), Y[ids].to(device)

@torch.no_grad()
def evaluate(model, ids, size, shift=0):
    model.eval(); L, P = 0.0, []
    for i in range(0, len(ids), 256):
        xb, yb = get_eval_batch(size, ids[i:i + 256], shift)
        out = model(xb); L += F.cross_entropy(out, yb, reduction="sum").item(); P.append(out.argmax(1).cpu())
    P = torch.cat(P).numpy()
    return L / len(ids), accuracy_score(labels[ids], P), P

def make_model(cfg):
    if cfg["model"] == "mlp": return MLP(cfg["size"])
    return CNN(cfg["size"], cfg["n_blocks"], cfg["filters"], cfg["kernel"], cfg.get("use_bn", True), cfg["dropout"])

def fit(cfg, seed):
    set_seed(seed); model = make_model(cfg).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"]); size = cfg["size"]
    Xtr, Ytr = train_set(size, cfg["aug"])
    h = dict(train_loss=[], train_acc=[], val_loss=[], val_acc=[], epoch_time=[])
    best, best_ep, best_state, bad = 1e9, 0, None, 0
    for ep in range(MAX_EPOCHS):
        t0 = time.time(); model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), BATCH):
            ids = perm[i:i + BATCH]
            if len(ids) < 2: continue
            opt.zero_grad(); F.cross_entropy(model(norm_batch(Xtr[ids], size)), Ytr[ids].to(device)).backward(); opt.step()
        if device.type == "cuda": torch.cuda.synchronize()
        h["epoch_time"].append(time.time() - t0)
        trl, tra, _ = evaluate(model, idx_tr, size)       # train ORIGINAL en modo eval (sin dropout ni copias aumentadas)
        vl, va, _ = evaluate(model, idx_va, size)
        h["train_loss"].append(trl); h["train_acc"].append(tra); h["val_loss"].append(vl); h["val_acc"].append(va)
        if vl < best - 1e-4: best, best_ep, best_state, bad = vl, ep, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    h.update(best_epoch=best_ep + 1, n_epochs=len(h["val_loss"]), n_params=n_params(model))
    return model, h

def run(cfg, seeds=SEEDS):
    # entrena la config con varias semillas y resume (media ± desvío)
    ms, hs = [], []
    for s in seeds:
        m, h = fit(cfg, s); ms.append(m); hs.append(h)
    be = [h["best_epoch"] - 1 for h in hs]
    va = [h["val_acc"][e] for h, e in zip(hs, be)]
    r = dict(val_acc=np.mean(va), val_acc_std=np.std(va), train_acc=np.mean([h["train_acc"][e] for h, e in zip(hs, be)]),
             val_loss=np.mean([min(h["val_loss"]) for h in hs]), best_epoch=np.mean([h["best_epoch"] for h in hs]),
             t_epoca_s=np.mean([np.mean(h["epoch_time"]) for h in hs]), t_total_s=np.mean([sum(h["epoch_time"]) for h in hs]), params=hs[0]["n_params"])
    r["brecha"] = r["train_acc"] - r["val_acc"]
    return r, ms, hs

BASE = dict(model="cnn", size=64, n_blocks=3, filters=16, kernel=3, dropout=0.2, lr=3e-3, aug="ninguna")
print("Parámetros CNN base (64px):", n_params(CNN(64)), "| MLP (64px):", n_params(MLP(64)), "| campo receptivo CNN base:", rf_full(3, 3), "px")

### 4.1 Cómo se transforma una imagen dentro de la CNN
La tabla muestra, capa por capa, la **forma de la salida** `(canales, alto, ancho)` y la **cantidad de parámetros** de la CNN base con una imagen de 64×64.

In [ ]:
def trace(model, size):
    model.eval(); x = torch.zeros(1, 3, size, size); rows = []
    for i, l in enumerate(list(model.features) + list(model.head)):
        x = l(x); rows.append(dict(capa=i, tipo=l.__class__.__name__, detalle=str(l)[:46], salida=tuple(x.shape[1:]), params=sum(p.numel() for p in l.parameters())))
    return pd.DataFrame(rows)
tr = trace(CNN(64), 64); display(tr); print("Total de parámetros:", tr["params"].sum())

### ✏️ EXPLICÁ EL CÓDIGO CONVOLUCIONAL
Con la clase `CNN` y la tabla de arriba, expliquen con sus palabras (y verificando con los números de la tabla):
1. **`padding=kernel // 2`**: ¿qué pasa con el alto y el ancho al aplicar la convolución? ¿Qué pasaría con `padding=0`? ¿Cómo se ve en la tabla?
2. **`MaxPool2d(2)`**: ¿qué hace con la forma de salida y qué información conserva? ¿Qué relación tiene con `fs = size // 2 ** n_blocks`?
3. **Parámetros:** calculen a mano cuántos parámetros tiene la segunda `Conv2d` (16 → 32 canales, kernel 3) y la primera `Linear` de la cabeza. Verifiquen con la tabla. ¿Cuál de las dos domina el total y por qué?
4. **`bias=not use_bn`:** ¿por qué la convolución no necesita sesgo cuando le sigue una BatchNorm?
5. **`filters * 2 ** b`:** ¿por qué se duplican los canales mientras se reduce la resolución a la mitad?
6. **`rf_full`:** reproduzcan a mano el campo receptivo de 18 px para 3 bloques con kernel 3 y expliquen qué significa ese número respecto de una imagen de 64 px.

1. Agrega un borde de ceros alrededor de la imagen para que la convolución no la achique. En la tabla se ve que las tres convs mantienen el tamaño (64x64, 32x32 y 16x16). Con padding=0 cada conv perdería 2 píxeles (64 pasaría a 62), las imágenes se irían achicando y el tamaño final ya no coincidiría con la entrada de la Linear.

2. Reduce a la mitad el alto y el ancho (64 a 32 a 16 a 8), quedándose con el valor más grande de cada cuadradito de 2x2. Conserva la activación más fuerte y pierde la posición exacta. Con fs = size // 2 ** n_blocks se calcula justo ese tamaño final, Hay un pooling por bloque, entonces 64/8 = 8, y por eso la salida es 64x8x8 = 4096 valores.

3. La segunda Conv2d tiene 16x32x3x3 = 4608 parámetros. La primera Linear tiene 4096x128 + 128 = 524.416. Los dos números coinciden con la tabla. Domina la Linear, porque cada una de las 4096 entradas se conecta con cada neurona, mientras que en la conv los mismos pesos se reutilizan en toda la imagen.

4. La BatchNorm que viene después ya resta la media y tiene su propio desplazamiento, así que el sesgo de la conv se cancelaría y no aportaría nada. Sacarlo ahorra parámetros.

5. Cada pooling achica la imagen, entonces se aumentan los canales para compensar. La red pasa de detectar cosas simples como bordes a combinaciones más complejas, y necesita más filtros distintos para eso. Además el tamaño total de las activaciones baja a la mitad por bloque (16.384 a 8192 a 4096), lo que mantiene el cómputo parecido entre capas.

6. Primer bloque: la conv suma 2 (queda 3) y el pooling suma 1 (queda 4). Segundo bloque: la conv suma 4 (8) y el pooling suma 2 (10). Tercer bloque: la conv suma 8 y da 18 y el último pooling no se cuenta. Eso quiere decir que cada neurona de la última conv “ve” un parche de 18x18 píxeles de la imagen original, o sea un 28% del lado de una imagen de 64 px. Alcanza para detectar bordes y texturas de la lesión, pero no la lesión entera. La visión global la arma la capa Linear.

### 4.2 Chequeo rápido: el modelo aprende
Se entrena la CNN base con una semilla y se grafican curvas. Sirve para verificar que el pipeline funciona antes de correr los experimentos.

In [ ]:
def plot_curves(hists, title):
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    for name, h in hists.items():
        ax[0].plot(h["train_loss"], label=name); ax[1].plot(h["val_loss"], label=name); ax[2].plot(h["val_acc"], label=name)
    for a, t in zip(ax, ["loss train (original, modo eval)", "loss validación", "accuracy validación"]): a.set_title(t); a.set_xlabel("época"); a.grid(alpha=.3)
    ax[0].legend(fontsize=8); fig.suptitle(title); plt.tight_layout(); plt.show()
t0 = time.time(); _, h = fit(BASE, 0)
print(f"mejor época {h['best_epoch']} | val_acc {h['val_acc'][h['best_epoch']-1]:.3f} | train_acc {h['train_acc'][h['best_epoch']-1]:.3f} | {np.mean(h['epoch_time']):.2f} s/época | total {time.time()-t0:.0f}s")
plot_curves({"CNN base": h}, "Chequeo de la CNN base (1 semilla)")

## 5. De la MLP a la CNN
Misma resolución, mismo split, mismo criterio de parada, 3 semillas. Además de val_acc se mide:
- **Parámetros y tiempo por época.**
- **Robustez a traslaciones**: se evalúa el modelo ya entrenado sobre validación con imágenes **trasladadas** al azar hasta ±`SHIFT` px (la convolución es equivariante a traslaciones y con pooling aproximadamente invariante; la MLP no tiene esa propiedad).

In [ ]:
SHIFT = {32: 4, 64: 8}
rows, store = [], {}
for model_name, size in [("mlp", 32), ("mlp", 64), ("cnn", 32), ("cnn", 64)]:
    cfg = {**BASE, "model": model_name, "size": size}
    r, ms, hs = run(cfg)
    clean = [evaluate(m, idx_va, size)[1] for m in ms]; shifted = [np.mean([evaluate(m, idx_va, size, shift=SHIFT[size])[1] for _ in range(3)]) for m in ms]
    rows.append(dict(modelo=f"{model_name.upper()} {size}px", params=r["params"], val_acc=r["val_acc"], std=r["val_acc_std"], train_acc=r["train_acc"],
                     brecha=r["brecha"], mejor_epoca=r["best_epoch"], t_epoca_s=r["t_epoca_s"],
                     val_acc_trasladada=np.mean(shifted), caida_por_traslacion=np.mean(clean) - np.mean(shifted)))
    store[f"{model_name} {size}"] = hs[0]
display(pd.DataFrame(rows).round(4))
plot_curves(store, "MLP vs CNN (semilla 0)")
print(f"Referencia: azar = {1/NUM_CLASSES:.3f} | error estándar binomial de un accuracy ~0.5 con {len(idx_va)} imágenes de val = {np.sqrt(.25/len(idx_va)):.3f}")

### ✏️ TU ANÁLISIS 1 - De la MLP a la CNN
1. Compará MLP y CNN en **parámetros, val_acc (± desvío), brecha train-val, tiempo por época y época de convergencia**. ¿La diferencia de val_acc supera el ruido entre semillas y el error estándar de la validación?
2. ¿Por qué la MLP necesita tantos más parámetros? Calculá a mano cuántos parámetros tiene la primera capa de la MLP a 32 y a 64 px y explicá qué pasaría a 224 px. ¿Qué propiedades de la convolución (pesos compartidos, conectividad local) cambian ese número?
3. Mirá la **caída por traslación**: ¿cuál de los dos modelos se degrada menos y por qué? ¿Es coherente con la teoría de equivarianza/invarianza, o el ruido no permite concluir?
4. ¿Qué rol cumple la normalización de entrada acá (mencionen cómo se calculó) y qué esperarían si no se hiciera?

1. La CNN obtiene una mejor val_acc que la MLP en ambos tamaños de imagen, con una diferencia de aproximadamente 12–13 puntos, que supera la variabilidad entre semillas y el error estándar de validación. Además, tiene muchos menos parámetros, lo que indica que aprovecha mejor la estructura espacial de las imágenes.

Ambos modelos presentan sobreajuste, pero la brecha train-val es menor en la CNN, por lo que generaliza mejor. También se observa que la MLP aumenta considerablemente su val_loss, mientras que la CNN se mantiene más estable.

La MLP converge antes, pero la CNN tarda aproximadamente tres veces más por época debido al costo de aplicar filtros sobre distintas posiciones de la imagen. Entre las CNN de 32 y 64 px no se observa una diferencia significativa de accuracy.

2. La MLP conecta cada píxel con todas las neuronas de su primera capa. Por eso, la cantidad de parámetros crece con la cantidad de píxeles.
- 32 px: 3072x512+512= 1.573.376 parámetros.
- 64 px: 12.288x512+512= 6.291.968 parámetros.
- 224 px: tendría más de 77 millones de parámetros solo en la primera capa.
En cambio, la CNN utiliza conectividad local y pesos compartidos. Esto permite detectar características sin multiplicar los parámetros al aumentar la resolución.

3. Los cuatro modelos disminuyen su accuracy al trasladar las imágenes, pero las diferencias en cuánto se degradan son pequeñas y están dentro de la variabilidad entre semillas. Por lo tanto, no se puede concluir que uno sea más robusto que otro.

Esto es coherente con la teoría: la convolución es equivariante a traslaciones, pero no garantiza invariancia en la clasificación final. El pooling aporta cierta invariancia local, aunque las capas posteriores siguen dependiendo de la posición. Además, los desplazamientos utilizados son relativamente grandes y el roll circular introduce píxeles de un borde en el opuesto, generando imágenes diferentes a las del entrenamiento.

4. Se restó la media y se dividió por el desvío estándar de cada canal RGB, calculados únicamente con las imágenes de entrenamiento. Esos mismos valores se utilizaron en validación y test para evitar filtraciones de información.

Sin normalizar, las entradas quedarían entre 0 y 1, pero sin estar centradas, lo que podría dificultar o volver menos estable el entrenamiento, especialmente en la MLP. En la CNN, BatchNorm puede compensarlo parcialmente. Sin embargo, esto es una expectativa teórica, ya que no se evaluó sin normalización.

## 6. Decisiones de arquitectura convolucional
Se parte de la CNN base y se cambia **un factor a la vez** (3 semillas cada uno): profundidad, tamaño del kernel, cantidad de filtros y resolución de entrada. Se reporta además el **campo receptivo** teórico y los parámetros (que incluyen la capa `Linear` de la cabeza, cuyo tamaño depende de la resolución y la profundidad).

In [ ]:
ARCH_VARIANTS = {
    "base (3 bloques, k=3, 16 filtros, 64px)": {},
    "2 bloques": dict(n_blocks=2), "4 bloques": dict(n_blocks=4),
    "kernel 5": dict(kernel=5), "kernel 7": dict(kernel=7),
    "8 filtros": dict(filters=8), "32 filtros": dict(filters=32),
    "32 px": dict(size=32), "96 px": dict(size=96),
}
rows, arch_cfgs, arch_hist = [], {}, {}
for name, delta in ARCH_VARIANTS.items():
    cfg = {**BASE, **delta}; r, ms, hs = run(cfg)
    rows.append(dict(variante=name, campo_receptivo=rf_full(cfg["n_blocks"], cfg["kernel"]), **r)); arch_cfgs[name] = cfg; arch_hist[name] = hs[0]
tab_arch = pd.DataFrame(rows)
display(tab_arch[["variante", "campo_receptivo", "params", "val_acc", "val_acc_std", "train_acc", "brecha", "best_epoch", "t_epoca_s", "t_total_s"]].round(4))
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(tab_arch["variante"], tab_arch["val_acc"], xerr=tab_arch["val_acc_std"], color="#4c72b0", alpha=.85); ax.invert_yaxis()
ax.axvline(tab_arch["val_acc"].iloc[0], color="k", ls="--", lw=1); ax.set_xlabel("val_acc (media ± desvío de 3 semillas)"); ax.set_title("Un factor a la vez respecto de la CNN base"); plt.tight_layout(); plt.show()

### ✏️ TU ANÁLISIS 2 - Arquitectura convolucional
1. ¿Qué factores cambian la val_acc **más allá del ruido entre semillas** (desvío de la tabla) y cuáles no? Sean explícitos sobre qué diferencias son pequeñas respecto del ruido.
2. **Profundidad y kernel:** relacionen el campo receptivo con el tamaño de la imagen y con lo que se quiere detectar en una lesión (bordes, texturas, asimetría global). ¿Qué esperaban de agregar bloques o agrandar el kernel y qué se observó? Comparen costo (parámetros y tiempo) vs. ganancia.
3. **Resolución de entrada (32/64/96):** ¿cómo cambia la cantidad de parámetros de la capa `Linear` de la cabeza y por qué? ¿Qué se gana (detalle disponible) y qué se pierde (cómputo, campo receptivo relativo a la imagen)? Con tan pocas imágenes por clase, ¿qué límite tiene este experimento?
4. ¿Qué no se puede concluir con un diseño de "un factor a la vez" y 3 semillas?

1. Ninguna variante supera claramente a la CNN base. Las que quedan por debajo de la base más allá del ruido entre semillas son pasar a 2 bloques y subir el kernel a 5: bajan unos 4 o 5 puntos y sus desvíos son de unos 2 puntos. Aun así, esa caída es parecida al error estándar de la validación, que con tan pocas imágenes ronda los 4 puntos, así que no es una conclusión firme. Todo lo demás cae dentro del ruido. El kernel 7, los 32 filtros y la resolución de 96 px bajan entre 2 y 4 puntos, pero con desvíos de 3 a 5 puntos, así que no se distinguen de la base. Los 8 filtros y los 32 px quedan a uno o dos puntos y empatan con la base. Los 4 bloques son un punto y medio mejores, pero eso también es menor que el desvío, así que es un empate y no una mejora. Lo único claro es que la base ya está en la zona alta y que agrandar el modelo no ayuda.

2. El campo receptivo de la base es de 18 px sobre una imagen de 64, suficiente para bordes y texturas pero no para la forma completa de la lesión. Con 2 bloques baja a 8 px y la red solo ve detalles muy locales. Con 4 bloques sube a 38 px, más de la mitad de la imagen, y con kernel 7 llega a 46 px. Esperábamos que ver más contexto ayudara a captar la asimetría y los bordes irregulares, que son lo que más importa en una lesión. Lo observado va en esa dirección solo parcialmente: 2 bloques empeora y 4 bloques mejora un poco, pero dentro del ruido. En el kernel no se ve ningún beneficio, ya que 5 y 7 empeoran. En costo, agrandar el campo receptivo con profundidad sale mucho más barato que hacerlo con kernels grandes. Los 4 bloques tienen menos parámetros que la base porque el mapa final es más chico y la Linear se achica, y el tiempo sube moderadamente. El kernel 7 en cambio multiplica el tiempo por unas cuatro veces y suma parámetros sin ninguna ganancia. Los 2 bloques tienen el doble de parámetros que la base por la misma razón inversa. La Linear recibe un mapa más grande. Además la base muestra la menor brecha entre train y val, y las variantes con más campo receptivo sobreajustan más, lo que es esperable con tan pocas imágenes.

3. La Linear de la cabeza recibe todos los valores del último mapa, y ese mapa crece con el cuadrado del lado de la imagen. A 32 px la cabeza es chica, a 64 es intermedia y a 96 se vuelve la parte dominante del modelo, que llega a más de un millón de parámetros. Lo que se gana con más resolución es detalle fino, como texturas o el borde de la lesión. Lo que se pierde es cómputo, porque el tiempo por época sube mucho, y campo receptivo relativo: como el campo receptivo es fijo en 18 px, cubre más de la mitad de la imagen a 32 px y solo una parte chica a 96, así que la red ve más local y menos global. En los resultados 96 px no mejora y 32 px empata con 64, o sea que el detalle extra no se aprovechó. El límite del experimento es que con unas 60 imágenes por clase una cabeza más grande tiende a memorizar en vez de aprender, y que además la validación es chica. Puede que una resolución baja alcance porque buena parte de la señal sea color o fondo y no textura fina, según se vio en el EDA. Eso es una hipótesis y no algo que se haya probado.

4. No se puede concluir nada sobre interacciones entre factores. Tampoco se puede saber si algunas variantes empeoran solo porque el learning rate y el dropout quedaron fijos y les convendría otros valores. Con 3 semillas el desvío es una estimación pobre, y las semillas solo cambian la inicialización y el orden de los datos, no la partición, así que todo se mide sobre la misma validación de pocas imágenes. Además la época de parada se elige con esa misma validación, lo que da un pequeño sesgo optimista. Por todo eso, el orden entre variantes cercanas no es confiable. En particular, que 4 bloques salga primero y sea el que se elija automáticamente para la configuración final no significa que sea realmente mejor que la base, y conviene decirlo así en el análisis final.

## 7. Data augmentation
Con ~60 imágenes por clase en train, el overfitting es esperable. Se comparan cuatro niveles de augmentation (los de la sección 3) con la CNN base y 3 semillas. En cada caso, **train queda con las imágenes originales más `AUG_COPIES` copias aumentadas de cada una**; val y test no se tocan.

In [ ]:
rows, aug_hist = [], {}
for name in AUGS:
    cfg = {**BASE, "aug": name}; r, ms, hs = run(cfg)
    rows.append(dict(augmentation=name, n_train=len(train_set(64, name)[0]), **r)); aug_hist[name] = hs[0]
tab_aug = pd.DataFrame(rows)
display(tab_aug[["augmentation", "n_train", "val_acc", "val_acc_std", "train_acc", "brecha", "val_loss", "best_epoch", "t_epoca_s", "t_total_s"]].round(4))
plot_curves(aug_hist, "Efecto de la data augmentation (semilla 0)")

### ✏️ TU ANÁLISIS 3 - Overfitting y augmentation
1. Demostrá el overfitting de la CNN sin augmentation **con números**: brecha train-val, época donde la `val_loss` empieza a subir.
2. ¿Qué cambia con cada nivel de augmentation sobre la **brecha**, la `val_loss`, la `val_acc`, la **época de convergencia** y el **tiempo total**? ¿Por qué cada época cuesta más con augmentation? ¿Cuál de las mejoras supera el ruido entre semillas?
3. Discutan desde el dominio (dermatología) qué transformaciones tienen sentido y cuáles podrían **destruir información** (por ejemplo, el color o la escala). ¿Qué otras augmentations probarían?
4. ¿Por qué es importante que la augmentation se haga **después** del split y **solo en train**? ¿Qué pasaría con las métricas si se aumentara antes de dividir?
5. Compará con lo que vieron en el TP3: ¿qué diferencia hay entre regularizar con dropout/L2 (restricciones sobre los pesos) y con augmentation (más datos sintéticos)?

1. Sin augmentation la CNN aprende mejor el train que la validación: llega a un 78% de accuracy en train contra un 64% en val, una brecha de unos 14 puntos. En las curvas se ve el patrón típico: la loss de train sigue bajando hasta quedar cerca de 0.35, mientras que la val_loss toca su mínimo, alrededor de 1.0, hacia la época 8 o 9 y después se vuelve inestable y sube hasta cerca de 1.5. Coincide con que la mejor época promedio es la 10 o 11 y que el early stopping corta poco después. No es un sobreajuste extremo, pero la red ya empieza a memorizar el train, y la validación deja de mejorar mucho antes de que el train termine de bajar.

2. Con flips la brecha baja, pero solo porque el accuracy de train cae, mientras que la val_acc también baja un poco, de modo que no es una mejora real. Converge mucho antes, hacia la época 6. Con flips más rot90 la val_acc queda igual que sin augmentation y la val_loss mejora apenas, pero la brecha sube porque el modelo ajusta mejor el train. Con color agregado pasa algo parecido: la brecha es la más alta, la val_loss la más baja de todas y la val_acc vuelve a quedar en el nivel de flips. Todas las diferencias de val_acc son de pocos puntos, menores o parecidas al desvío entre semillas y al error estándar de la validación, así que ninguna augmentation supera el ruido. Las diferencias de val_loss son todavía más chicas y la tabla no trae desvío para esa columna, por lo que tampoco se las toma como mejora. Cada época cuesta tres veces más porque train pasa de 506 a 1518 imágenes, el original más dos copias aumentadas, y la red tiene que procesar el triple de datos. El tiempo total sube entre dos veces y media y tres veces, y más aún con rot90 y color, que además entrenan más épocas.

3. Los flips y las rotaciones de 90 grados tienen sentido porque una lesión dermatoscópica no tiene una orientación privilegiada y la foto se puede tomar con cualquier ángulo. El color en cambio es información diagnóstica, porque el pigmento de un melanoma o el rojo de una lesión vascular son parte de lo que distingue las clases, así que cambios fuertes de brillo o contraste pueden confundir al modelo. Una variación leve como la que usamos simula diferencias de iluminación, pero no ayudó. La escala también es delicada porque el tamaño de la lesión es un criterio clínico y un zoom o recorte agresivo puede borrar información o cortar la lesión. Otras augmentations para probar serían rotaciones de ángulo libre con bordes reflejados, recortes y zooms suaves, un poco de desenfoque o ruido para imitar distintas cámaras, borrado aleatorio de parches y mixup. También convendría generar las transformaciones en cada época y no una sola vez, porque con copias fijas la red ve siempre las mismas dos variantes y puede memorizarlas, lo que probablemente limita el efecto que se observa.

4. Si se aumentara antes de dividir, las versiones volteadas o rotadas de una misma imagen caerían en train y también en val o test. El modelo vería en evaluación imágenes casi iguales a las que ya vio, igual que pasaba con los duplicados, y las métricas quedarían infladas y no medirían generalización. Además se usaria esa validación contaminada para elegir el modelo, y esa elección sería mala. Por eso se divide primero y se aumenta solo train. Val y test tienen que mostrar imágenes reales no vistas, sin transformar, porque eso es lo que se va a encontrar el modelo en uso.
Dropout y L2 actúan sobre el modelo: restringen los pesos o apagan neuronas para que no se ajuste demasiado, pero no agregan información sobre los datos. La augmentation actúa sobre los datos: agrega ejemplos sintéticos que le enseñan al modelo una invarianza que conocemos del dominio, como que una lesión sigue siendo la misma si se la gira, y deja el modelo intacto. La primera es barata y genérica, y la segunda cuesta más cómputo pero puede aportar conocimiento del problema. En los resultados la CNN base ya trae dropout y BatchNorm, y sumarle augmentation no dio una mejora medible, así que con tan pocas imágenes y con un posible atajo por fuente o color, las transformaciones geométricas por sí solas no alcanzan.

## 8. Elección del modelo final y evaluación en TEST (una sola vez)
Se arma la configuración final tomando, entre las variantes de arquitectura, la de **mejor val_acc media** y, entre las de augmentation, la de **mejor val_acc media** (ambas ya medidas arriba). Se puede forzar otra decisión con `FINAL_OVERRIDE` (justificándola en el análisis). Es una decisión *greedy* de un factor a la vez: no se garantiza que la combinación sea mejor que cada parte.

Luego se entrenan la config final y la MLP de referencia (64 px) con las semillas `SEEDS_FINAL` y se evalúan en test **una única vez**. No se vuelve a tocar nada después de ver estos números.

In [ ]:
best_arch = tab_arch.sort_values(["val_acc", "params"], ascending=[False, True]).iloc[0]["variante"]
best_aug  = tab_aug.sort_values("val_acc", ascending=False).iloc[0]["augmentation"]
FINAL_CFG = {**arch_cfgs[best_arch], "aug": best_aug}
FINAL_OVERRIDE = None          # por ejemplo: dict(n_blocks=4, aug="flips+rot90")
if FINAL_OVERRIDE: FINAL_CFG = {**FINAL_CFG, **FINAL_OVERRIDE}
print("Mejor arquitectura (val):", best_arch, "| mejor augmentation (val):", best_aug)
print("Config final:", FINAL_CFG)
rf_, ms_, hs_ = run(FINAL_CFG)
print(f"Config final en validación: val_acc {rf_['val_acc']:.3f} ± {rf_['val_acc_std']:.3f} | brecha {rf_['brecha']:.3f} | mejor época {rf_['best_epoch']:.1f} | {rf_['t_total_s']:.0f}s por entrenamiento")

In [ ]:
# ---- EVALUACIÓN EN TEST: se ejecuta una sola vez ----
final_rows, final_models, final_preds = [], {}, {}
for name, cfg in [("CNN final", FINAL_CFG), ("MLP 64px", {**BASE, "model": "mlp", "size": 64})]:
    accs, f1s, times = [], [], []
    for s in SEEDS_FINAL:
        m, h = fit(cfg, s); _, a, p = evaluate(m, idx_te, cfg["size"])
        accs.append(a); f1s.append(f1_score(labels[idx_te], p, average="macro")); times.append(sum(h["epoch_time"]))
        if s == SEEDS_FINAL[0]: final_models[name], final_preds[name] = m, p
    final_rows.append(dict(modelo=name, params=h["n_params"], test_acc=np.mean(accs), test_acc_std=np.std(accs), test_f1_macro=np.mean(f1s), f1_std=np.std(f1s), tiempo_entrenamiento_s=np.mean(times)))
final = pd.DataFrame(final_rows); display(final.round(4))
se = np.sqrt(final.loc[0, "test_acc"] * (1 - final.loc[0, "test_acc"]) / len(idx_te))
print(f"Error estándar binomial del test ({len(idx_te)} imágenes): ±{se:.3f} | azar: {1/NUM_CLASSES:.3f}")
p = final_preds["CNN final"]
print(classification_report(labels[idx_te], p, target_names=class_names, zero_division=0))
cm = confusion_matrix(labels[idx_te], p)
plt.figure(figsize=(7, 6)); plt.imshow(cm, cmap="Blues"); plt.colorbar()
plt.xticks(range(NUM_CLASSES), class_names, rotation=90); plt.yticks(range(NUM_CLASSES), class_names)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES): plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xlabel("Predicción"); plt.ylabel("Real"); plt.title(f"Matriz de confusión - CNN final (test, semilla {SEEDS_FINAL[0]})"); plt.tight_layout(); plt.show()

## 9. ¿Qué aprende la red?
Se muestran (sobre imágenes de **validación**, no de test): los filtros de la primera convolución, los **mapas de activación** de la primera y la última capa convolucional para una imagen, y una galería de imágenes **mal clasificadas**.

In [ ]:
model = final_models["CNN final"].eval(); size = FINAL_CFG["size"]
W = model.features[0].weight.detach().cpu(); n = min(W.shape[0], 16)
fig, ax = plt.subplots(1, n, figsize=(1.2 * n, 1.5))
for i in range(n):
    w = W[i]; w = (w - w.min()) / (w.max() - w.min() + 1e-8); ax[i].imshow(w.permute(1, 2, 0), interpolation="nearest"); ax[i].axis("off")
fig.suptitle("Filtros de la primera convolución (kernel de %dx%d, 3 canales RGB)" % (W.shape[2], W.shape[3])); plt.show()

relu_pos = [i for i, l in enumerate(model.features) if isinstance(l, nn.ReLU)]
xi, yi = get_eval_batch(size, idx_va[:1])
with torch.no_grad(): a_first = model.features[:relu_pos[0] + 1](xi)[0].cpu(); a_last = model.features[:relu_pos[-1] + 1](xi)[0].cpu()
fig, ax = plt.subplots(2, 9, figsize=(14, 3.4))
for r_, (a, nm) in enumerate([(a_first, "1ª conv"), (a_last, "última conv")]):
    ax[r_, 0].imshow(X_U8[size][idx_va[0]].permute(1, 2, 0)); ax[r_, 0].set_title(nm, fontsize=8); ax[r_, 0].axis("off")
    for k in range(8): ax[r_, k + 1].imshow(a[k], cmap="viridis"); ax[r_, k + 1].axis("off"); ax[r_, k + 1].set_title(f"canal {k} ({a.shape[1]}x{a.shape[2]})", fontsize=7)
plt.suptitle("Mapas de activación (después de ReLU): primeros 8 canales"); plt.show()

_, _, pv = evaluate(model, idx_va, size)
bad = [i for i, p_ in zip(idx_va, pv) if p_ != labels[i]][:10]; pbad = [p_ for i, p_ in zip(idx_va, pv) if p_ != labels[i]][:10]
fig, ax = plt.subplots(2, 5, figsize=(12, 5.4))
for a, i, pr in zip(ax.flat, bad, pbad):
    a.imshow(X_U8[96][i].permute(1, 2, 0)); a.axis("off"); a.set_title(f"real: {class_names[labels[i]][:16]}\npred: {class_names[pr][:16]}", fontsize=8, color="red")
plt.suptitle("Primeras 10 imágenes de validación mal clasificadas"); plt.tight_layout(); plt.show()

### ✏️ TU ANÁLISIS 4 - Resultados finales e interpretación
1. **Consistencia val vs. test:** ¿el resultado de test es coherente con la validación? Tengan en cuenta el error estándar binomial del test y que elegimos el mejor entre varias configuraciones sobre validación (sesgo optimista).
2. **CNN vs. MLP en test:** ¿la diferencia supera el ruido entre semillas (± desvío) y el error estándar del test? ¿Se justifica el costo en parámetros/tiempo?
3. **Matriz de confusión y reporte por clase:** ¿qué clases se confunden más y por qué (similitud clínica/visual)? En este problema, ¿es igual de grave confundir cualquier par de clases? ¿Qué métrica (recall de melanoma, F1 macro, otra) les parece más relevante que el accuracy y por qué?
4. **Filtros, mapas de activación y errores:** ¿qué detectan las primeras capas y qué cambia en la última (tamaño espacial, nivel de abstracción)? ¿Qué patrones comunes tienen las imágenes mal clasificadas?
5. **Sesgos y limitaciones:** retomen el EDA (resoluciones y color por clase, duplicados). ¿Qué sesgos del dataset podrían inflar o distorsionar los números? Tamaño del dataset, una sola partición, pocas semillas, validación chica: ¿qué harían con más tiempo?

1. El resultado de test es coherente con la validación. La config final llegó a una accuracy de 0.677 con desvío de 0.020 entre 5 semillas, y en validación las CNN rondaban entre 0.62 y 0.65. Test queda unos pocos puntos por encima, pero la diferencia es menor que el error estándar binomial de test, que es de 0.036, así que no hay una diferencia real entre los dos. No se nota el sesgo optimista por haber elegido la mejor configuración sobre validación, aunque con tan pocas imágenes tampoco podríamos detectarlo. Además, como vimos en el Análisis 2, la ventaja de los 4 bloques sobre la base estaba dentro del ruido, así que esa elección fue en buena parte azar. Por eso lo razonable es leer el resultado como una accuracy de alrededor de 0.65 a 0.70 y no como un número exacto.

2. La CNN supera claramente a la MLP: 0.677 contra 0.501 de accuracy y 0.668 contra 0.482 de F1 macro. La diferencia, de unos 18 puntos, es mucho mayor que el desvío entre semillas de ambos modelos y que el error estándar del test, y además la MLP es bastante más inestable entre semillas, con un desvío de 0.060 contra 0.020. En costo, la CNN usa unas 17 veces menos parámetros, pero tarda unas 10 veces más en entrenar, unos 30 segundos contra 3. Ese tiempo extra es irrelevante en la práctica y la mejora es grande, así que se justifica sin dudas.

3. Las clases que mejor anda son Vascular lesion, Benign keratosis y Atopic Dermatitis, con recall de 1.00, 0.95 y 0.94. Las peores son Tinea, Melanoma y Squamous cell carcinoma, con 0.38 y 0.45 las dos últimas. Las confusiones más fuertes son Dermatofibroma y Squamous cell carcinoma clasificadas como Actinic keratosis, lo que explica que Actinic tenga una precisión de solo 0.56 aunque casi nunca se le escape una, y también Melanocytic nevus confundido con Melanoma. Son lesiones rosadas y de bajo contraste que se parecen entre sí, y un nevus y un melanoma pueden ser visualmente muy cercanos, incluso para una persona. Los melanomas, además, se reparten en muchas clases distintas. No es igual de grave confundir cualquier par: tomar un melanoma por una lesión benigna, como un nevus o una queratosis benigna, puede retrasar un diagnóstico serio, mientras que lo contrario genera una biopsia innecesaria. Por eso la métrica más relevante es el recall de melanoma, que acá es de 0.45, o sea que casi la mitad de los melanomas no se detectan. El F1 macro también es mejor que la accuracy, porque da el mismo peso a todas las clases, y con pocas imágenes por clase la accuracy global tapa a las clases débiles.

4. Los filtros de la primera convolución son parches de 3x3 con colores variados y no se parecen a detectores de bordes limpios, lo que indica que están mirando sobre todo color y contraste local. En los mapas de la primera capa, que tienen el tamaño completo de la imagen, algunos canales quedan apagados y otros resaltan la zona pigmentada con bastante detalle, siguiendo la forma de la lesión. En la última capa los mapas son de 8x8, muy gruesos, varios canales están casi vacíos y los que se activan marcan zonas amplias, sin forma reconocible. Eso es lo esperado, porque la red pasa de detalles locales a señales más abstractas y de menor resolución. Entre las imágenes mal clasificadas hay varios patrones. Muchas son primeros planos sin un borde claro de la lesión, o lesiones rosadas y de bajo contraste que se parecen a otras clases. Otras son melanomas muy oscuros o rojizos que el modelo toma por una lesión vascular o un nevus, o sea errores guiados por el color. Varios casos serían difíciles incluso para un ojo humano.

5. El principal problema es el que se vio en el EDA: la resolución casi identifica la clase, porque todas las imágenes de Atopic Dermatitis tienen el mismo tamaño y son fotos de piel y no dermatoscopías, y los colores medios difieren bastante entre clases. Es probable que parte del buen resultado en Atopic Dermatitis y Vascular lesion no venga de la lesión sino de la fuente de la imagen o del color, y como test sale de las mismas fuentes, el modelo no se enfrenta a un dominio distinto y los números quedan inflados. Se eliminaron los duplicados exactos, pero pueden quedar imágenes distintas de una misma lesión o paciente repartidas entre train, val y test, que no se detectaron. Además el dataset es muy chico. Test tiene unas 20 imágenes por clase y 13 en Tinea, así que el recall de cada clase tiene mucha incertidumbre. Se usó una sola partición, 5 semillas que solo cambian la inicialización, y una validación chica que además se usó para elegir el modelo. Con más tiempo se podria hacer validación cruzada estratificada, más semillas, un split que agrupe por paciente o lesión si el dato existe, y una prueba con imágenes de otra fuente. También se podria probar una pérdida que dé más peso a las clases difíciles como melanoma, transfer learning con una red preentrenada y mapas de atención tipo Grad-CAM para ver si la red mira la lesión o el fondo.

## Declaración de uso de IA
Completen (obligatorio): ¿usaron herramientas de IA? ¿Cuáles, para qué parte (código, redacción del análisis, ideas) y qué modificaron o verificaron por su cuenta? Si no usaron, aclaren que no.

> *Usé herramientas de IA, Claude y ChatGPT, con dos fines. Por un lado, para entender algunas partes del código y para discutir e interpretar mejor los resultados. Por otro lado, como ayuda de redacción: yo escribía primero lo que quería decir en cada análisis y después usaba la IA para ordenar y redactar el texto, porque me cuesta expresarme por escrito y quería que se entendiera bien lo que quería transmitir.*